# 03: AUC-ROC comparison

Ten-seed AUC-ROC comparison of the PGNN with Random Forest and Gradient Boosting on the sweep-grouped split, with paired t-tests (Table 5, AUC column).

**Run:** place `impedance.csv` and `frequencies.csv` in `data/` (or upload them when prompted in Colab), then Runtime → Run all.

## 0. Setup and data

In [ ]:

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, f1_score
from scipy import stats
import random, json, time, os

t0 = time.time()

IMP_PATH, FREQ_PATH = "data/impedance.csv", "data/frequencies.csv"
if not (os.path.exists(IMP_PATH) and os.path.exists(FREQ_PATH)):
    os.makedirs("data", exist_ok=True)
    try:
        from google.colab import files
        print("Please upload impedance.csv and frequencies.csv:")
        uploaded = files.upload()
        for fname in uploaded:
            target = "data/" + fname
            with open(target, "wb") as f:
                f.write(uploaded[fname])
            print("saved", target)
    except ImportError:
        raise FileNotFoundError(
            "data/impedance.csv and data/frequencies.csv not found, and this "
            "doesn't look like Colab. Place both files under a data/ folder "
            "next to this notebook and re-run this cell."
        )
print("Data files present.")


Please upload impedance.csv and frequencies.csv:


Saving frequencies (2).csv to frequencies (2).csv
Saving impedance (2).csv to impedance (2).csv
saved data/frequencies (2).csv
saved data/impedance (2).csv
Data files present.


## 1. Preprocessing, corrected split, model (identical to the main Experiments notebook)

In [ ]:

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

impedance = pd.read_csv(IMP_PATH)
frequencies = pd.read_csv(FREQ_PATH)
df = impedance.merge(frequencies, on="FREQUENCY_ID", how="left")
df["SWEEP_ID"] = (df["BATTERY_ID"].astype(str) + "_" + df["MEASURE_ID"].astype(str) + "_" + df["SOC"].astype(str))

def parse_complex(s):
    if pd.isna(s): return np.nan
    s = str(s).replace(" ", "").replace("(", "").replace(")", "")
    try: return complex(s)
    except ValueError: return np.nan

df["Z_complex"] = df["IMPEDANCE_VALUE"].apply(parse_complex)
df["Re_Z"] = df["Z_complex"].apply(lambda z: z.real)
df["Im_Z"] = df["Z_complex"].apply(lambda z: z.imag)
df["abs_Z"] = df["Z_complex"].apply(lambda z: abs(z))
df = df.dropna(subset=["Re_Z", "Im_Z", "abs_Z", "SOC"]).reset_index(drop=True)
freq_col = "FREQUENCY_VALUE"
df["log10_f"] = np.log10(df[freq_col].astype(float))
df["phi"] = np.arctan2(df["Im_Z"], df["Re_Z"])
SOC_THRESHOLD = 60
df["label"] = (df["SOC"] >= SOC_THRESHOLD).astype(int)
FEATS = ["log10_f", "Re_Z", "Im_Z", "abs_Z", "phi"]

def make_windows(group):
    g = group.sort_values(freq_col).reset_index(drop=True)
    rows, labels, zcentre = [], [], []
    n = len(g)
    for i in range(1, n - 1):
        vec = np.concatenate([g.loc[i-1, FEATS].values, g.loc[i, FEATS].values, g.loc[i+1, FEATS].values]).astype(float)
        rows.append(vec); labels.append(g.loc[i, "label"]); zcentre.append(g.loc[i, "abs_Z"])
    return rows, labels, zcentre

X, y, z, sweep_ids = [], [], [], []
for key, g in df.groupby("SWEEP_ID", sort=False):
    rows, labels, zc = make_windows(g)
    X.extend(rows); y.extend(labels); z.extend(zc)
    sweep_ids.extend([key] * len(rows))
X, y, z, sweep_ids = np.array(X), np.array(y), np.array(z), np.array(sweep_ids)
print(f"n={len(X)} windows (expect 2,880).")

def group_boundary_indices(sids):
    boundaries = [0]; prev = sids[0]
    for i in range(1, len(sids)):
        if sids[i] != prev:
            boundaries.append(i); prev = sids[i]
    boundaries.append(len(sids))
    return boundaries

n = len(X)
bounds = group_boundary_indices(sweep_ids)
n_train = min(bounds, key=lambda b: abs(b - 0.70 * n))
later = [b for b in bounds if b > n_train]
n_trainval = min(later, key=lambda b: abs(b - 0.85 * n))
for cut in (n_train, n_trainval):
    assert sweep_ids[cut - 1] != sweep_ids[cut], f"cut at {cut} splits a sweep!"

X_tr, y_tr, z_tr = X[:n_train], y[:n_train], z[:n_train]
X_val, y_val, z_val = X[n_train:n_trainval], y[n_train:n_trainval], z[n_train:n_trainval]
X_te, y_te, z_te = X[n_trainval:], y[n_trainval:], z[n_trainval:]
print(f"Corrected split: train={len(X_tr)}, val={len(X_val)}, test={len(X_te)}")

def normalise_z(z_train, *others):
    zmin, zmax = z_train.min(), z_train.max()
    def norm(zz): return np.clip((zz - zmin) / (zmax - zmin + 1e-12), 0, 1)
    return (norm(z_train),) + tuple(norm(o) for o in others)

scaler = StandardScaler().fit(X_tr)
X_tr_s, X_val_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_val), scaler.transform(X_te)
z_tr_n, z_val_n, z_te_n = normalise_z(z_tr, z_val, z_te)

class PGNN(nn.Module):
    def __init__(self, in_dim=15):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,128), nn.Tanh(), nn.Dropout(0.2),
            nn.Linear(128,128), nn.Tanh(), nn.Dropout(0.2), nn.Linear(128,64), nn.Tanh(), nn.Linear(64,2))
    def forward(self, x): return self.net(x)

def physics_loss(p1, z_norm):
    mask = (z_norm > 0.5).float()
    return torch.mean(p1 * z_norm * mask)

def train_pgnn(X_train, y_train, z_train, X_val, y_val, lam=1.0, in_dim=15,
               lr=5e-4, max_epochs=300, patience=25, batch_size=64, seed=42):
    torch.manual_seed(seed)
    Xt = torch.tensor(X_train, dtype=torch.float32); yt = torch.tensor(y_train, dtype=torch.long)
    zt = torch.tensor(z_train, dtype=torch.float32)
    Xv = torch.tensor(X_val, dtype=torch.float32); yv = torch.tensor(y_val, dtype=torch.long)
    model = PGNN(in_dim)
    ce = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=lr, betas=(0.9,0.999), weight_decay=1e-4)
    sched = optim.lr_scheduler.StepLR(opt, step_size=50, gamma=0.5)
    n_ = len(Xt); best_val_loss, best_state, bad_epochs = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train(); perm = torch.randperm(n_)
        for i in range(0, n_, batch_size):
            idx = perm[i:i+batch_size]; xb, yb, zb = Xt[idx], yt[idx], zt[idx]
            opt.zero_grad(); logits = model(xb); probs = torch.softmax(logits, dim=1)
            loss = ce(logits, yb) + lam * physics_loss(probs[:,1], zb)
            loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            val_loss = ce(model(Xv), yv).item()
        if val_loss < best_val_loss - 1e-6:
            best_val_loss, best_state, bad_epochs = val_loss, {k:v.clone() for k,v in model.state_dict().items()}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience: break
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_proba(model, X):
    model.eval(); Xt = torch.tensor(X, dtype=torch.float32)
    return torch.softmax(model(Xt), dim=1)[:,1].numpy()

print(f"Setup done ({time.time()-t0:.0f}s elapsed).")


n=2880 windows (expect 2,880).
Corrected split: train=2016, val=432, test=432
Setup done (14s elapsed).



## 2. AUC-ROC comparison, 10 seeds each

Same `random_state`/`seed` convention as `pinn_vs_baselines.json` (seeds
0-9), same corrected split, same RF/GB hyperparameters. PINN uses
`predict_proba` on the test set directly (no thresholding needed for AUC).
Paired t-test, PINN vs. each baseline, on AUC-ROC.


In [ ]:

N_SEEDS = 10
pinn_aucs, rf_aucs, gb_aucs = [], [], []
t1 = time.time()
for seed in range(N_SEEDS):
    model = train_pgnn(X_tr_s, y_tr, z_tr_n, X_val_s, y_val, lam=1.0, seed=seed)
    p1_te = predict_proba(model, X_te_s)
    pinn_aucs.append(roc_auc_score(y_te, p1_te))

    rf = RandomForestClassifier(n_estimators=300, criterion="gini", class_weight="balanced", random_state=seed)
    rf.fit(X_tr_s, y_tr)
    rf_proba = rf.predict_proba(X_te_s)[:, 1]
    rf_aucs.append(roc_auc_score(y_te, rf_proba))

    gb = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.1, random_state=seed)
    gb.fit(X_tr_s, y_tr)
    gb_proba = gb.predict_proba(X_te_s)[:, 1]
    gb_aucs.append(roc_auc_score(y_te, gb_proba))

    print(f"seed {seed}: PINN AUC={pinn_aucs[-1]:.4f}  RF AUC={rf_aucs[-1]:.4f}  GB AUC={gb_aucs[-1]:.4f}  "
          f"({time.time()-t1:.0f}s elapsed)")

pinn_aucs, rf_aucs, gb_aucs = np.array(pinn_aucs), np.array(rf_aucs), np.array(gb_aucs)
t_rf, p_rf = stats.ttest_rel(pinn_aucs, rf_aucs)
t_gb, p_gb = stats.ttest_rel(pinn_aucs, gb_aucs)

auc_comparison = {
    "pinn_mean_auc": float(pinn_aucs.mean()), "pinn_sd_auc": float(pinn_aucs.std()),
    "rf_mean_auc": float(rf_aucs.mean()), "rf_sd_auc": float(rf_aucs.std()),
    "gb_mean_auc": float(gb_aucs.mean()), "gb_sd_auc": float(gb_aucs.std()),
    "pinn_vs_rf_delta": float(pinn_aucs.mean() - rf_aucs.mean()), "pinn_vs_rf_pvalue": float(p_rf),
    "pinn_vs_gb_delta": float(pinn_aucs.mean() - gb_aucs.mean()), "pinn_vs_gb_pvalue": float(p_gb),
    "n_seeds": N_SEEDS,
}
print()
print("Summary:", auc_comparison)

with open("pinn_vs_baselines_auc.json", "w") as f:
    json.dump(auc_comparison, f, indent=2)
print("Saved pinn_vs_baselines_auc.json")
print(f"Total elapsed: {time.time()-t0:.0f}s")


seed 0: PINN AUC=0.8676  RF AUC=0.8816  GB AUC=0.8975  (15s elapsed)
seed 1: PINN AUC=0.9366  RF AUC=0.8803  GB AUC=0.8966  (29s elapsed)
seed 2: PINN AUC=0.9159  RF AUC=0.8745  GB AUC=0.8970  (43s elapsed)
seed 3: PINN AUC=0.8998  RF AUC=0.8833  GB AUC=0.8970  (61s elapsed)
seed 4: PINN AUC=0.9039  RF AUC=0.8787  GB AUC=0.8973  (77s elapsed)
seed 5: PINN AUC=0.9161  RF AUC=0.8827  GB AUC=0.8972  (93s elapsed)
seed 6: PINN AUC=0.9223  RF AUC=0.8792  GB AUC=0.8965  (109s elapsed)
seed 7: PINN AUC=0.8917  RF AUC=0.8800  GB AUC=0.8964  (123s elapsed)
seed 8: PINN AUC=0.9339  RF AUC=0.8805  GB AUC=0.8976  (140s elapsed)
seed 9: PINN AUC=0.9019  RF AUC=0.8807  GB AUC=0.8970  (160s elapsed)

Summary: {'pinn_mean_auc': 0.9089691840277776, 'pinn_sd_auc': 0.019588571002852648, 'rf_mean_auc': 0.8801367187499999, 'rf_sd_auc': 0.002316206224253093, 'gb_mean_auc': 0.8970030381944445, 'gb_sd_auc': 0.00039185281573008205, 'pinn_vs_rf_delta': 0.02883246527777772, 'pinn_vs_rf_pvalue': 0.002067255317579


## 3. Download

The result is saved as `pinn_vs_baselines_auc.json`.

In [ ]:

try:
    from google.colab import files
    files.download("pinn_vs_baselines_auc.json")
except ImportError:
    pass


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>